# Complete Databricks ↔ Azure ML Integration Guide

This notebook provides a comprehensive overview of **how to integrate Databricks with Azure ML** for production data science workflows.

## Integration Scenarios
1. **Databricks → Azure ML**: Use Databricks for data prep, train in AML
2. **Azure ML → Databricks**: Use AML for training, batch scoring in Databricks
3. **Bidirectional**: Model registry sync, shared data sources
4. **Real-time Inference**: Call AML endpoints from Databricks
5. **Auto-retraining**: AML triggers Databricks pipelines

## Architecture Overview
```
┌─────────────────────────────────────────────────────────┐
│              Azure Databricks Workspace                 │
│  ┌──────────────────────────────────────────────────┐  │
│  │  Unity Catalog (Governed Data Layer)             │  │
│  │  - bronze tables (raw data)                      │  │
│  │  - silver tables (cleaned features)              │  │
│  │  - gold tables (ML-ready)                        │  │
│  │  - Model Registry                                │  │
│  └──────────────────────────────────────────────────┘  │
└────────────────┬────────────────────────────┬───────────┘
                 │ (1) Export features        │ (4) Deploy models
                 │ (2) Read models in batch   │ (5) Score in DBX
                 ▼                            ▼
          ┌──────────────────────────────────────┐
          │     Azure Databricks Storage         │
          │  (ADLS Gen2 + Unity Catalog Vol)     │
          └──────────────────────────────────────┘
                 ▲
                 │ Managed Identity
                 │ (No shared keys)
                 ▼
          ┌──────────────────────────────────────┐
          │       Azure ML Workspace             │
          │  ┌──────────────────────────────┐   │
          │  │  Model Training              │   │
          │  │  - Experiments               │   │
          │  │  - Runs & metrics            │   │
          │  │  - Model Registry            │   │
          │  └──────────────────────────────┘   │
          │  ┌──────────────────────────────┐   │
          │  │  Endpoints                   │   │
          │  │  - Real-time (online)        │   │
          │  │  - Batch (offline)           │   │
          │  └──────────────────────────────┘   │
          └──────────────────────────────────────┘
```

## 1. Databricks → Azure ML: Feature Engineering & Training

### Scenario 1A: Export Databricks Features to Azure ML

**Workflow:**
1. Run feature engineering in Databricks (Spark jobs)
2. Store features in Unity Catalog (governed, versioned)
3. Export to Azure Blob/ADLS (via UC external volumes)
4. Azure ML reads data for training

**Why this matters:**
- ✅ Databricks excels at distributed feature engineering (Spark)
- ✅ Unity Catalog provides data governance & lineage
- ✅ Azure ML focuses on model training & serving
- ✅ Clean separation of concerns

In [ ]:
# STEP 1: Feature Engineering in Databricks
# (This runs in Databricks cluster)

from pyspark.sql import functions as F
from pyspark.sql.types import *
import datetime

# Configuration
UC_CATALOG = "prod"
UC_SCHEMA = "ml_features"
FEATURES_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.features_v1"
FEATURE_VOLUME = f"/{UC_CATALOG}/{UC_SCHEMA}/volumes/aml_input"
AML_STORAGE_PATH = "abfss://aml-training@yourstg.dfs.core.windows.net/features/"

# Example: Create features in Databricks
features_df = spark.sql(f"""
SELECT 
    customer_id,
    total_purchases,
    avg_order_value,
    days_since_last_purchase,
    churn_flag,
    CURRENT_TIMESTAMP() as feature_timestamp,
    '{datetime.date.today().strftime('%Y%m%d')}' as feature_version
FROM dev.raw.customer_data
WHERE date >= DATE_SUB(CURRENT_DATE(), 90)
""")

print(f"Features count: {features_df.count()}")
features_df.show(5)

In [ ]:
# STEP 2: Store in Unity Catalog (Governed)

# Create UC table
features_df.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(FEATURES_TABLE)

print(f"✓ Features stored in UC table: {FEATURES_TABLE}")

# Verify
spark.sql(f"SELECT COUNT(*) as feature_count FROM {FEATURES_TABLE}").show()

# Grant permissions
spark.sql(f"GRANT SELECT ON TABLE {FEATURES_TABLE} TO `account users`")
print(f"✓ Permissions granted")

In [ ]:
# STEP 3: Export to Azure Storage (for AML to read)

# Option A: Export to Excel/CSV for small datasets
features_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(AML_STORAGE_PATH)

print(f"✓ Features exported to: {AML_STORAGE_PATH}")

# Option B: Export as Parquet (more efficient)
features_df.write \
    .mode("overwrite") \
    .parquet(AML_STORAGE_PATH.replace("/", "").replace(".", "/") + "parquet")

print(f"✓ Features exported as Parquet")

### Scenario 1B: Azure ML Reads Databricks Features

This runs in **Azure ML** (not Databricks):

```python
# In Azure ML training script (train.py)
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.storage.blob import BlobServiceClient

# Read exported features from Databricks
credential = DefaultAzureCredential()
blob_service = BlobServiceClient(
    account_url="https://yourstg.blob.core.windows.net",
    credential=credential
)

# Download features
blob_client = blob_service.get_blob_client(
    container="aml-training",
    blob="features/part-00000.csv"
)

with open("features.csv", "wb") as f:
    f.write(blob_client.download_blob().readall())

# Load and train
df = pd.read_csv("features.csv")
model = train_model(df)
```

## 2. Azure ML → Databricks: Model Scoring & Batch Predictions

### Scenario 2A: Call Azure ML Endpoint from Databricks

**Use when:**
- Model trained in AML, deployed on endpoint
- Need batch or real-time scoring from Databricks
- Want to score millions of records efficiently

In [ ]:
# STEP 1: Call AML endpoint from Databricks

import requests
import json
from azure.identity import DefaultAzureCredential, ChainedTokenCredential
import logging

logger = logging.getLogger(__name__)

# Configuration
AML_ENDPOINT_URL = "https://my-endpoint.eastus.inference.ml.azure.com/score"
AML_ENDPOINT_KEY = dbutils.secrets.get(scope="aml_creds", key="endpoint_key") if "dbutils" in dir() else "your-key"

# Get authentication token
credential = DefaultAzureCredential()
token = credential.get_token("https://ml.azure.com/.default").token

# Prepare scoring data
input_data = {
    "dataframe_split": {
        "columns": ["feature1", "feature2", "feature3"],
        "data": [
            [1.0, 2.0, 3.0],
            [4.0, 5.0, 6.0]
        ]
    }
}

# Call AML endpoint
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

response = requests.post(
    AML_ENDPOINT_URL,
    headers=headers,
    json=input_data,
    timeout=30
)

if response.status_code == 200:
    predictions = response.json()
    print(f"✓ Predictions received: {predictions}")
else:
    print(f"✗ Error: {response.status_code} - {response.text}")

In [ ]:
# STEP 2: Batch Scoring with Spark UDF
# Score millions of records efficiently using Spark distributed processing

from pyspark.sql.functions import pandas_udf, col, struct
import pandas as pd

@pandas_udf("double")
def score_with_aml_endpoint(features: pd.Series) -> pd.Series:
    """
    Spark UDF to call AML endpoint for batch scoring.
    This runs in parallel across Databricks cluster workers.
    """
    import requests
    from azure.identity import DefaultAzureCredential
    
    credential = DefaultAzureCredential()
    token = credential.get_token("https://ml.azure.com/.default").token
    
    predictions = []
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    for feature_row in features:
        input_data = {"dataframe_split": {"data": [feature_row]}}
        try:
            response = requests.post(
                AML_ENDPOINT_URL,
                headers=headers,
                json=input_data,
                timeout=30
            )
            if response.status_code == 200:
                pred = response.json()[0]  # Extract prediction
                predictions.append(pred)
            else:
                predictions.append(None)
        except Exception as e:
            print(f"Error scoring: {e}")
            predictions.append(None)
    
    return pd.Series(predictions)

# Apply to DataFrame
features_df = spark.sql(f"SELECT * FROM {FEATURES_TABLE}")

scoring_df = features_df.withColumn(
    "prediction",
    score_with_aml_endpoint(
        struct(col("total_purchases"), col("avg_order_value"), col("days_since_last_purchase"))
    )
)

print("✓ Batch scoring complete")
scoring_df.show(5)

## 3. Model Registry Synchronization

### Scenario 3A: Register Models in Both Registries

**You need BOTH registries because:**
- 📊 **Databricks Model Registry** - version models, track lineage from UC data
- 🤖 **Azure ML Model Registry** - deploy endpoints, A/B testing, ops features

**Workflow:**
1. Train in Azure ML → save model
2. Register in Azure ML Model Registry
3. Download model artifact
4. Register in Databricks Model Registry
5. Deploy from either location

In [ ]:
# STEP 1: Register model in Databricks Model Registry

import mlflow
from mlflow.tracking import MlflowClient
import joblib

# Set MLflow tracking URI (Databricks workspace)
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/youruser/ml_experiments/churn_prediction")

# Assume you have a trained model
# model = train_your_model(features_df)

with mlflow.start_run(run_name="churn_model_v1") as run:
    # Log parameters
    mlflow.log_params({
        "algorithm": "RandomForest",
        "n_estimators": 100,
        "max_depth": 10
    })
    
    # Log metrics
    mlflow.log_metrics({
        "accuracy": 0.92,
        "precision": 0.88,
        "recall": 0.85,
        "f1_score": 0.865
    })
    
    # Log model artifact
    mlflow.sklearn.log_model(
        # model,  # Uncomment when you have actual model
        artifact_path="model",
        registered_model_name="ChurnPredictionModel"
    )
    
    run_id = run.info.run_id

print(f"✓ Model registered in Databricks (run_id: {run_id})")

In [ ]:
# STEP 2: Sync Databricks model to Azure ML

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.identity import DefaultAzureCredential
import os

# Initialize AML client
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=os.getenv("AZURE_SUBSCRIPTION_ID"),
    resource_group_name=os.getenv("AZURE_RESOURCE_GROUP"),
    workspace_name=os.getenv("AZURE_WORKSPACE_NAME")
)

# Register model in Azure ML
model = Model(
    path="dbfs:/databricks/model-registry/ChurnPredictionModel/",  # DBX model path
    name="churn-prediction-model",
    description="Churn prediction model trained on Databricks UC features",
    type="custom_model",
    properties={
        "source": "databricks",
        "framework": "sklearn",
        "databricks_registered_model": "ChurnPredictionModel"
    }
)

registered_model = ml_client.models.create_or_update(model)
print(f"✓ Model registered in Azure ML: {registered_model.id}")

## 4. Real-Time Inference: Databricks Model Serving vs AML Endpoints

### Scenario 4A: Databricks Model Serving (Recommended for UC Models)

**Advantages:**
- Direct UC integration (models + data lineage)
- Fast inference (co-located with data)
- Built for Spark-optimized models
- Simpler governance

In [ ]:
# Call Databricks Model Serving Endpoint

import requests
import json

DATABRICKS_HOST = "https://adb-123.azuredatabricks.net"
DATABRICKS_TOKEN = dbutils.secrets.get(scope="databricks", key="pat_token")
ENDPOINT_NAME = "churn-prediction"  # Created by Bicep/Terraform

# Prepare input
input_data = {
    "dataframe_split": {
        "columns": ["total_purchases", "avg_order_value", "days_since_last_purchase"],
        "data": [[1000, 50, 30]]
    }
}

# Call endpoint
headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

url = f"{DATABRICKS_HOST}/serving-endpoints/{ENDPOINT_NAME}/invocations"

response = requests.post(url, headers=headers, json=input_data)

if response.status_code == 200:
    predictions = response.json()
    print(f"✓ Prediction: {predictions}")
else:
    print(f"✗ Error: {response.text}")

### Scenario 4B: Azure ML Endpoint (For Enterprise Deployment)

**Advantages:**
- Enterprise monitoring & compliance
- A/B testing, traffic routing
- Sophisticated security (private endpoints, managed identity)
- Cost tracking per endpoint

In [ ]:
# Deploy & Call Azure ML Endpoint

from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    CodeConfiguration,
    Environment,
    Model
)
from azure.identity import DefaultAzureCredential
import requests

# Initialize client
ml_client = MLClient(credential=DefaultAzureCredential())

# Create endpoint
endpoint = ManagedOnlineEndpoint(
    name="churn-prediction-ep",
    description="Churn prediction endpoint",
    auth_mode="key"
)

endpoint = ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"✓ Endpoint created: {endpoint.scoring_uri}")

# Deploy model
deployment = ManagedOnlineDeployment(
    name="churn-model-v1",
    endpoint_name="churn-prediction-ep",
    model=Model(name="churn-prediction-model", version="1"),
    code_configuration=CodeConfiguration(
        code="./src",
        scoring_script="score.py"
    ),
    environment=Environment(name="sklearn-staging", version="1"),
    instance_type="Standard_DS2_v2",
    instance_count=2
)

deployment = ml_client.online_deployments.begin_create_or_update(deployment).result()
print(f"✓ Deployment created: {deployment.name}")

# Get credentials
endpoint_creds = ml_client.online_endpoints.get_keys("churn-prediction-ep")
print(f"Primary key: {endpoint_creds.primary_key}")

## 5. Automated Retraining Pipeline

### Scenario 5A: AML Trigger → Databricks Retraining

**Workflow:**
1. AML monitors model performance (accuracy, drift detection)
2. If metrics degrade, trigger Databricks job
3. Databricks retrains using fresh features from UC
4. Register new model, evaluate, promote

In [ ]:
# Trigger Databricks Job from Azure ML

import subprocess
import json
import time

# Using Databricks CLI (must be installed)
DATABRICKS_JOB_ID = 123  # Create this job via Terraform

# Trigger job
cmd = [
    "databricks",
    "jobs",
    "run-now",
    "--job-id", str(DATABRICKS_JOB_ID),
    "--output", "json"
]

result = subprocess.run(cmd, capture_output=True, text=True)
run_info = json.loads(result.stdout)
run_id = run_info["run_id"]

print(f"✓ Databricks job triggered (run_id: {run_id})")

# Poll for completion
status = "PENDING"
while status not in ["TERMINATED", "SKIPPED", "INTERNAL_ERROR"]:
    cmd = [
        "databricks",
        "jobs",
        "get-run",
        "--run-id", str(run_id),
        "--output", "json"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    job_info = json.loads(result.stdout)
    status = job_info["state"]
    print(f"Job status: {status}")
    time.sleep(30)

result_state = job_info.get("result_state", "UNKNOWN")
if result_state == "SUCCESS":
    print("✓ Retraining completed successfully")
else:
    print(f"✗ Retraining failed: {result_state}")

## 6. Authentication Patterns

### Pattern A: Managed Identity (Recommended)

```
Databricks Workspace
    ↓ (uses)
  Access Connector
    ↓ (has)
  Managed Identity
    ↓ (authenticates as)
  Azure Resources
    ├─ Storage (ADLS Gen2)
    ├─ Key Vault
    └─ Azure ML Workspace
```

**Setup by Bicep:**
- Creates Access Connector with managed identity
- Grants RBAC roles to managed identity
- No secrets need to be rotated

**Code usage:**
```python
credential = DefaultAzureCredential()
# Automatically uses managed identity from Access Connector
```

### Pattern B: Service Principal (For External Apps)

```
External App (GitHub Actions, Azure DevOps)
    ↓ (uses)
  Service Principal Credentials
    ├─ Client ID
    ├─ Client Secret
    └─ Tenant ID
    ↓ (authenticates to)
  Azure Resources
```

**Code usage:**
```python
from azure.identity import ClientSecretCredential

credential = ClientSecretCredential(
    tenant_id=os.getenv("AZURE_TENANT_ID"),
    client_id=os.getenv("AZURE_CLIENT_ID"),
    client_secret=os.getenv("AZURE_CLIENT_SECRET")
)
```

## 7. Data Flow Architecture

### Data Flow 1: Feature Engineering Path

```
Raw Data (ADLS Gen2)
    ↓
Databricks Bronze Table (UC)
    ↓
Feature Engineering Spark Job
    ↓
Databricks Silver Table (UC) ← Features
    ↓
Export to Databricks Gold Table
    ↓
Export to ADLS (for AML)
    ↓
Azure ML Training Job reads CSV/Parquet
    ↓
Train Model (scikit-learn, TensorFlow, etc.)
    ↓
Register in AML + Databricks Model Registry
    ↓
Deploy to Endpoint (DBX or AML)
```

### Data Flow 2: Inference Path

```
Scoring Requests
    ├─→ Databricks Endpoint (Low latency, UC models)
    │       ↓
    │   Model Serving
    │       ↓
    │   Predictions
    │
    └─→ Azure ML Endpoint (Enterprise, compliance)
            ↓
        Managed Endpoint
            ↓
        Predictions
```

## 8. Comparison: When to Use What

| Task | Databricks | Azure ML | Why |
|------|-----------|----------|-----|
| **Feature Engineering** | ✅✅✅ | ❌ | Spark distributed processing is best |
| **Data Governance** | ✅✅✅ | ❌ | Unity Catalog provides lineage & control |
| **Model Training** | ✅✅ | ✅✅✅ | AML has better experiment tracking |
| **Hyperparameter Tuning** | ✅ | ✅✅✅ | AML has AutoML & Hyperdrive |
| **Model Serving** | ✅✅ | ✅✅✅ | Both good; DBX if low latency needed |
| **Real-time Inference** | ✅✅ | ✅✅✅ | Near-equivalent; AML better for compliance |
| **Batch Scoring** | ✅✅✅ | ✅ | Databricks excels at scale |
| **Monitoring/Observability** | ✅ | ✅✅✅ | AML has more sophisticated tools |
| **A/B Testing** | ✅ | ✅✅✅ | AML has native traffic routing |
| **Pipeline Orchestration** | ✅✅ | ✅✅ | Use Azure DevOps or Airflow |

**General Recommendation:**
- 🎯 **Databricks for:** Data prep, feature engineering, batch scoring, cost optimization
- 🎯 **Azure ML for:** Experiment tracking, model training, governance, enterprise deployment

## 9. Integration Checklist

### Pre-Deployment (Bicep/Terraform)

- [ ] Create Databricks workspace with VNet injection
- [ ] Create Azure ML workspace in same region
- [ ] Create ADLS Gen2 storage account (HNS enabled)
- [ ] Create Access Connector with managed identity
- [ ] Assign RBAC roles (Contributor to managed identity)
- [ ] Create UC metastore (Quebec region)
- [ ] Enable private endpoints for network security
- [ ] Create Key Vault for secrets

### Post-Deployment (Notebooks)

- [ ] Test Databricks CLI connectivity
- [ ] Test Azure ML CLI connectivity
- [ ] Create UC catalogs & schemas
- [ ] Run feature engineering pipeline
- [ ] Export features to ADLS
- [ ] Train model in Azure ML
- [ ] Register model in both registries
- [ ] Deploy Databricks Model Serving endpoint
- [ ] Deploy Azure ML managed endpoint
- [ ] Test end-to-end scoring
- [ ] Set up monitoring & alerting
- [ ] Document connection strings

### Production (Ongoing)

- [ ] Monitor model performance (accuracy, drift)
- [ ] Retrain on schedule or on-demand
- [ ] Track lineage (data → features → model → predictions)
- [ ] Audit access via UC
- [ ] Cost optimization (cluster sizing, auto-shutdown)

## 10. Summary: 5-Step Integration Process

```
STEP 1: Prepare Data
    └─→ Databricks: Feature engineering with Spark
        └─→ Store in Unity Catalog
        └─→ Export to ADLS

STEP 2: Train Models
    └─→ Azure ML: Read prepared features
        └─→ Train with AutoML or custom code
        └─→ Register in AML Model Registry

STEP 3: Register Models
    └─→ Databricks: Register same model to DBX Registry
        └─→ Track lineage to UC source data

STEP 4: Deploy
    └─→ Choose endpoint:
        ├─→ Databricks Model Serving (low latency + data governance)
        └─→ Azure ML Managed Endpoint (enterprise compliance)

STEP 5: Score & Monitor
    └─→ Call endpoint for predictions
        └─→ Monitor performance metrics
        └─→ Trigger retraining when needed
        └─→ Go back to STEP 1
```